# Deadline patterns → `insights`

Finds which deadline patterns come before cohort stress spikes, per course and week, and writes plain-language findings to the `insights` table (shown in the rep insights panel, R6 / issue #22).

**Privacy (docs/requirements.md §5):** this notebook only reads `daily_course_stress` rows with `n_students >= 5`, plus `deadlines` and `courses`. It never reads `checkins`, `checkin_course`, user ids, or raw text, and it doesn't call Gemini.

**How to run**
1. Run the `%pip` cell once, then `dbutils.library.restartPython()` if `import pg8000` fails. The notebook uses `pg8000` (pure Python) because `psycopg[binary]` crashes the Serverless kernel on connect.
2. Set the widgets at the top:
   - `SOURCE`: `synthetic` (built-in demo data, the default) or `postgres` (Tiger Data).
   - `WRITE`: `false` (dry run, the default) or `true`. Findings are only written when `SOURCE = postgres`.
   - `DATABASE_URL`: only used if the secret below isn't set. **Clear this widget before exporting the notebook to Git.**
3. Run all.

**Preferred credential:** a Databricks secret `stress-forecast / DATABASE_URL`. The role needs `SELECT` on `daily_course_stress`, `deadlines` and `courses`, plus `SELECT`, `INSERT` and `DELETE` on `insights`.

In [ ]:
%pip install pg8000

## 1. Settings

In [ ]:
import datetime as dt
import os

import numpy as np
import pandas as pd

# Must match backend/app/config.py
COHORT_MIN = 5                 # COHORT_MIN
HEAVY_WEIGHT_PCT = 40          # COLLISION_MIN_WEIGHT
PRE_DEADLINE_DAYS = 3          # COLLISION_PEAK_LOOKBACK_DAYS

# Notebook-only analysis thresholds
MAJOR_WEIGHT_PCT = 20          # a week with this much assessment weight due is a "major" week
TOP_PEAKS = 3                  # how many of the highest-stress weeks to examine
MIN_WEEKS = 4                  # skip courses with fewer weeks of visible data
MIN_RAMP = 1.0                 # min pre-deadline rise worth reporting
MIN_GAP = 0.5                  # min heavy-vs-other week gap worth reporting
LOCAL_TZ = "America/Vancouver"


def _widget(name, default, choices=None):
    """Databricks widget if available, otherwise an env var (for local runs)."""
    try:
        if choices:
            dbutils.widgets.dropdown(name, default, choices)  # noqa: F821
        else:
            dbutils.widgets.text(name, default)  # noqa: F821
        return dbutils.widgets.get(name) or default  # noqa: F821
    except NameError:
        return os.getenv(name, default)


def _database_url():
    """The secret if it exists, otherwise the DATABASE_URL widget."""
    try:
        url = dbutils.secrets.get("stress-forecast", "DATABASE_URL")  # noqa: F821
        if url:
            return url
    except Exception:
        pass
    url = _widget("DATABASE_URL", "").strip()
    if not url:
        raise ValueError("Paste the Tiger connection string into the DATABASE_URL widget at the top, then run again.")
    return url


SOURCE = _widget("SOURCE", "synthetic", ["synthetic", "postgres"])
WRITE = _widget("WRITE", "false", ["false", "true"]) == "true"
_widget("DATABASE_URL", "")  # create the box up front so it's visible before the first postgres run
print(f"SOURCE={SOURCE}  WRITE={WRITE}")

## 2. Load data

Deadlines are saved per student (`POST /deadlines` stores each student's reviewed copy), so the same assessment can appear many times. They are de-duplicated by course, title and due date, taking the most common title spelling and the median weight.

In [ ]:
def connect(url):
    """pg8000 connection from a postgres:// URL. Like sslmode=require: encrypted, certificate
    not verified (the Serverless egress path presents a self-signed certificate)."""
    import ssl
    from urllib.parse import unquote, urlsplit

    import pg8000.native

    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    u = urlsplit(url)
    return pg8000.native.Connection(
        user=unquote(u.username or ""), password=unquote(u.password or ""),
        host=u.hostname, port=u.port or 5432, database=u.path.lstrip("/") or None,
        ssl_context=ctx, timeout=30,
    )


def _fetch(conn, sql, **params):
    rows = conn.run(sql, **params)
    return pd.DataFrame(rows, columns=[c["name"] for c in conn.columns])


def load_postgres(conn):
    daily = _fetch(
        conn,
        """
        SELECT course_id,
               (day AT TIME ZONE CAST(:tz AS text))::date AS date,
               avg_stress::float8                         AS avg_stress,
               n_students
        FROM daily_course_stress
        WHERE n_students >= CAST(:min AS int)
        ORDER BY course_id, day
        """,
        tz=LOCAL_TZ, min=COHORT_MIN,
    )
    deadlines = _fetch(
        conn, "SELECT course_id, title, due_date, weight_pct::float8 AS weight_pct, type FROM deadlines"
    )
    courses = _fetch(conn, "SELECT id AS course_id, code, name FROM courses")
    return daily, deadlines, courses


def dedupe_deadlines(deadlines):
    if deadlines.empty:
        return deadlines.assign(n_reports=pd.Series(dtype=int))
    d = deadlines.assign(key=deadlines["title"].str.strip().str.lower())
    return (
        d.groupby(["course_id", "key", "due_date"], as_index=False)
        .agg(
            title=("title", lambda s: s.mode().iat[0]),
            weight_pct=("weight_pct", "median"),
            type=("type", lambda s: s.mode().iat[0]),
            n_reports=("title", "size"),
        )
        .drop(columns="key")
        .sort_values(["course_id", "due_date"], ignore_index=True)
    )

In [ ]:
def synthetic_data(seed=7):
    """Demo term matching the mock courses: stress rises 1-3 days before heavy deadlines,
    with a week 9 pile-up inside CPSC 210 and a few small-sample days that get filtered."""
    rng = np.random.default_rng(seed)
    courses = pd.DataFrame(
        [
            (1, "CPSC 210", "Software Construction"),
            (2, "MATH 200", "Calculus III"),
            (3, "PHYS 118", "Electricity, Light and Radiation"),
            (4, "ENGL 110", "Approaches to Literature"),
        ],
        columns=["course_id", "code", "name"],
    )
    rows = [
        (1, "Lab 1", "2026-09-18", 2, "assignment"),
        (1, "Project Phase 1", "2026-10-09", 10, "project"),
        (1, "Midterm 1", "2026-10-14", 20, "midterm"),
        (1, "Project Phase 2", "2026-10-30", 10, "project"),
        (1, "Midterm 2", "2026-11-03", 25, "midterm"),
        (1, "Project Phase 3", "2026-11-05", 15, "project"),
        (1, "Lab 6", "2026-11-06", 5, "assignment"),
        (1, "Lab 8", "2026-11-27", 2, "assignment"),
        (2, "WeBWorK 1", "2026-09-17", 2, "quiz"),
        (2, "Midterm 1", "2026-10-16", 20, "midterm"),
        (2, "WeBWorK 5", "2026-10-22", 2, "quiz"),
        (2, "Midterm 2", "2026-11-13", 20, "midterm"),
        (2, "WeBWorK 9", "2026-11-26", 2, "quiz"),
        (3, "Lab report 1", "2026-09-25", 5, "assignment"),
        (3, "Midterm", "2026-10-21", 25, "midterm"),
        (3, "Project", "2026-11-20", 20, "project"),
        (4, "Essay 1", "2026-10-02", 15, "assignment"),
        (4, "Essay 2", "2026-11-06", None, "assignment"),
        (4, "Essay 3", "2026-11-27", 20, "assignment"),
    ]
    deadlines = pd.DataFrame(rows, columns=["course_id", "title", "due_date", "weight_pct", "type"])
    deadlines["due_date"] = pd.to_datetime(deadlines["due_date"]).dt.date
    # Simulate several students saving the same deadline (one with a typo'd weight).
    deadlines = pd.concat([deadlines] * 3, ignore_index=True)
    deadlines.loc[0, "weight_pct"] = 3

    days = pd.date_range("2026-09-08", "2026-12-04").date
    out = []
    for cid in courses["course_id"]:
        dl = deadlines[deadlines["course_id"] == cid].drop_duplicates(["title", "due_date"])
        for day in days:
            stress = 4.2 + (0.4 if day.weekday() < 5 else -0.3)
            for _, d in dl.iterrows():
                lead = (d["due_date"] - day).days
                w = d["weight_pct"] if pd.notna(d["weight_pct"]) else 10
                if 0 <= lead <= PRE_DEADLINE_DAYS:
                    stress += w / 9 * (1 - lead / (PRE_DEADLINE_DAYS + 1))
                elif lead == -1:
                    stress -= 0.3
            stress = float(np.clip(stress + rng.normal(0, 0.35), 1, 10))
            n = int(rng.integers(18, 40)) if rng.random() > 0.06 else int(rng.integers(1, COHORT_MIN))
            out.append((cid, day, round(stress, 2), n))
    daily = pd.DataFrame(out, columns=["course_id", "date", "avg_stress", "n_students"])
    return daily, deadlines, courses


if SOURCE == "postgres":
    _conn = connect(_database_url())
    try:
        daily, raw_deadlines, courses = load_postgres(_conn)
    finally:
        _conn.close()
else:
    daily, raw_deadlines, courses = synthetic_data()

# Defensive second filter: never analyse suppressed days, whatever the source.
daily = daily[daily["n_students"] >= COHORT_MIN].reset_index(drop=True)
deadlines = dedupe_deadlines(raw_deadlines)
print(f"{len(daily)} visible course-days, {len(deadlines)} unique deadlines "
      f"(from {len(raw_deadlines)} saved rows), {len(courses)} courses")

## 3. Weekly features

Per course and week (Monday start): average and peak stress, number of deadlines due, their known total weight, and the change in average stress vs the previous week. A week is **heavy** when its known deadline weight reaches `HEAVY_WEIGHT_PCT` (unknown weights count as 0, so this is a lower bound).

In [ ]:
def week_start(dates):
    s = pd.to_datetime(pd.Series(dates))
    return (s - pd.to_timedelta(s.dt.day_of_week, unit="D")).dt.date


def weekly_table(daily, deadlines):
    d = daily.assign(week_start=week_start(daily["date"]).values)
    weeks = d.groupby(["course_id", "week_start"], as_index=False).agg(
        avg_stress=("avg_stress", "mean"),
        peak_stress=("avg_stress", "max"),
        visible_days=("avg_stress", "size"),
    )
    dl = deadlines.assign(week_start=week_start(deadlines["due_date"]).values)
    load = dl.groupby(["course_id", "week_start"], as_index=False).agg(
        deadline_count=("title", "size"),
        known_weight=("weight_pct", lambda s: s.fillna(0).sum()),
        titles=("title", lambda s: ", ".join(s)),
    )
    w = weeks.merge(load, on=["course_id", "week_start"], how="left")
    w = w.fillna({"deadline_count": 0, "known_weight": 0.0, "titles": ""})
    w["deadline_count"] = w["deadline_count"].astype(int)
    w = w.sort_values(["course_id", "week_start"], ignore_index=True)
    w["delta"] = w.groupby("course_id")["avg_stress"].diff()
    w["heavy"] = w["known_weight"] >= HEAVY_WEIGHT_PCT
    w["major"] = w["known_weight"] >= MAJOR_WEIGHT_PCT
    # A peak "lines up with deadlines" if a major week is this week or the next one.
    w["major_now_or_next"] = w["major"] | w.groupby("course_id")["major"].shift(-1, fill_value=False)
    return w


def pre_deadline_ramps(daily, deadlines):
    """Avg stress in the PRE_DEADLINE_DAYS before each deadline, vs the course's median day."""
    rows = []
    for cid, days in daily.groupby("course_id"):
        baseline = days["avg_stress"].median()
        by_date = days.set_index("date")["avg_stress"]
        for _, d in deadlines[deadlines["course_id"] == cid].iterrows():
            window = [d["due_date"] - dt.timedelta(days=k) for k in range(1, PRE_DEADLINE_DAYS + 1)]
            vals = by_date.reindex(window).dropna()
            if len(vals) >= 2:
                rows.append({
                    "course_id": cid, "title": d["title"], "due_date": d["due_date"],
                    "weight_pct": d["weight_pct"], "pre_avg": vals.mean(),
                    "baseline": baseline, "ramp": vals.mean() - baseline,
                })
    return pd.DataFrame(rows, columns=["course_id", "title", "due_date", "weight_pct", "pre_avg", "baseline", "ramp"])


weekly = weekly_table(daily, deadlines)
ramps = pre_deadline_ramps(daily, deadlines)
try:
    display(weekly)  # noqa: F821  (Databricks / Jupyter)
except NameError:
    print(weekly.head(15).to_string())

## 4. Findings

Up to three plain-language findings per course, each tied to the week it is about:
1. Heavy weeks vs other weeks (average stress gap).
2. How many of the highest-stress weeks fell in or just before a week with `MAJOR_WEIGHT_PCT`%+ due.
3. The deadline with the largest stress rise in the days before it.

In [ ]:
def _pct(x):
    return f"{x:g}%" if pd.notna(x) else "unknown weight"


def course_findings(cid, w, r):
    findings = []
    if len(w) < MIN_WEEKS:
        return findings

    heavy, other = w[w["heavy"]], w[~w["heavy"]]
    if len(heavy) >= 1 and len(other) >= 2:
        gap = heavy["avg_stress"].mean() - other["avg_stress"].mean()
        if gap >= MIN_GAP:
            top = heavy.loc[heavy["known_weight"].idxmax()]
            findings.append((top["week_start"],
                f"Weeks with assessments worth {HEAVY_WEIGHT_PCT}%+ averaged {heavy['avg_stress'].mean():.1f} "
                f"stress vs {other['avg_stress'].mean():.1f} in other weeks (+{gap:.1f})."))

    peaks = w.nlargest(TOP_PEAKS, "peak_stress")
    hits = int(peaks["major_now_or_next"].sum())
    if len(peaks) == TOP_PEAKS and hits >= 2:
        top = peaks.iloc[0]
        change = f", {top['delta']:+.1f} vs the week before" if pd.notna(top["delta"]) else ""
        findings.append((top["week_start"],
            f"{hits} of the {TOP_PEAKS} highest-stress weeks came in or just before a week with "
            f"{MAJOR_WEIGHT_PCT}%+ of assessments due. The highest daily average was "
            f"{top['peak_stress']:.1f} in the week of {top['week_start']:%b %-d}{change}."))

    if not r.empty:
        top = r.loc[r["ramp"].idxmax()]
        if top["ramp"] >= MIN_RAMP:
            findings.append((week_start([top["due_date"]]).iat[0],
                f"Average stress rose {top['ramp']:.1f} points above the term's typical level in the "
                f"{PRE_DEADLINE_DAYS} days before {top['title']} ({_pct(top['weight_pct'])}, due {top['due_date']:%b %-d})."))
    return findings


rows = []
for cid in sorted(weekly["course_id"].unique()):
    for week, text in course_findings(cid, weekly[weekly["course_id"] == cid], ramps[ramps["course_id"] == cid]):
        rows.append({"course_id": int(cid), "week_start": week, "finding": text})
findings = pd.DataFrame(rows, columns=["course_id", "week_start", "finding"])

codes = dict(zip(courses["course_id"], courses["code"]))
for _, f in findings.iterrows():
    print(f"[{codes.get(f['course_id'], f['course_id'])} | {f['week_start']}] {f['finding']}")

## 5. Write to `insights`

Skipped unless `WRITE = true` **and** `SOURCE = postgres`, so synthetic findings never reach the database. A run **replaces** the stored findings of every course it analysed (delete, then insert, in one transaction), so findings from older data or an older seed never linger. Running twice on the same data leaves the same rows.

In [ ]:
INSERT_SQL = """
INSERT INTO insights (course_id, week_start, finding)
SELECT CAST(:course_id AS int), CAST(:week_start AS date), CAST(:finding AS text)
WHERE NOT EXISTS (
    SELECT 1 FROM insights
    WHERE course_id = CAST(:course_id AS int)
      AND week_start = CAST(:week_start AS date)
      AND finding = CAST(:finding AS text)
)
"""


def write_findings(conn, findings, course_ids):
    """Replace the stored findings of ``course_ids`` with ``findings``.
    Returns (deleted, inserted). Caller wraps it in a transaction."""
    conn.run("DELETE FROM insights WHERE course_id = ANY(CAST(:ids AS int[]))", ids=sorted(int(c) for c in course_ids))
    deleted = conn.row_count
    inserted = 0
    for rec in findings.to_dict("records"):
        conn.run(INSERT_SQL, course_id=int(rec["course_id"]), week_start=rec["week_start"], finding=rec["finding"])
        inserted += conn.row_count
    return deleted, inserted


if WRITE and SOURCE == "postgres":
    _conn = connect(_database_url())
    try:
        _conn.run("START TRANSACTION")
        # Every course that was analysed, so a course with no findings this run is cleared too.
        deleted, inserted = write_findings(_conn, findings, courses["course_id"])
        _conn.run("COMMIT")
    except Exception:
        _conn.run("ROLLBACK")
        raise
    finally:
        _conn.close()
    print(f"Replaced {deleted} old finding(s) with {inserted} new one(s).")
else:
    print(f"Dry run: {len(findings)} finding(s) not written (WRITE={WRITE}, SOURCE={SOURCE}).")